# **Inferencia binaria: HybridCNN Audio Classifier (`alertable`)**

Carga el modelo entrenado y predice sobre audios `.wav` / `.mp3` usando **exactamente el mismo pipeline** que `preprocess.py → main()` a través de `Preprocess.process_audio_file()`.

Soporta los modos `mel_only`, `mel_mfcc` y `mel_waveform` con detección automática desde el state_dict.

> **`AUGMENT_INFERENCE`** — activa aumentaciones de dominio (ruido, EQ telefónico, reverb sintética) sobre audios limpios externos para acercarlos al dominio de entrenamiento. Ponlo en `True` cuando uses audios descargados de internet.

## 1 · Importaciones

In [1]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from src.utils.config import *
from src.models.hybrid_cnn_v3 import ImprovedMFCCCNN
from src.data.preprocess import Preprocess, PreprocessConfig  # ← pipeline centralizado

print("✅ Importaciones completadas")

✅ Importaciones completadas


## 2 · Configuración

In [47]:
# ─────────────────────────────────────────────
# 🔧 RUTAS
# ─────────────────────────────────────────────
MODEL_PATH         = FINAL_MODEL_DIR / "best_alertable_v3.pt"
MODEL_PATH       = CHECKPOINT_DIR / "alertable_V3" / "checkpoint_epoch_16.pt"

PROCESSED_METADATA = PROCESSED_METADATA['2']
LABEL_MAPPING_PATH = LABEL_MAPPING["alertable2"]

AUDIO_FOLDER       = TEST_AUDIO_FOLDER   # carpeta con .wav / .mp3

# ─────────────────────────────────────────────
# 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# ─────────────────────────────────────────────
SAMPLE_RATE    = 16000
N_MELS         = 128
N_MFCC         = 13
N_FFT          = 1024
HOP_LENGTH     = 160
PEAK_TARGET    = 0.99

# ─────────────────────────────────────────────
# 🧠 MODELO
# ─────────────────────────────────────────────
DROPOUT        = 0.25

# ─────────────────────────────────────────────
# 🖥️ INFERENCIA
# ─────────────────────────────────────────────
TOP_K          = 2       # binario: solo 2 clases

# ─────────────────────────────────────────────
# 🔊 AUGMENTACIÓN DE DOMINIO EN INFERENCIA
# ─────────────────────────────────────────────
# Ponlo en True cuando el audio venga de internet / estudio (limpio).
# Ponlo en False si el audio ya proviene del mismo dominio del dataset.
AUGMENT_INFERENCE = True

print(f"📂 Carpeta de audios : {AUDIO_FOLDER}")
print(f"📦 Modelo            : {MODEL_PATH}")
print(f"🗂️  Label mapping      : {LABEL_MAPPING_PATH}")

📂 Carpeta de audios : /home/andres/Documentos/proyecto4geeks/tests/audios
📦 Modelo            : /home/andres/Documentos/proyecto4geeks/models/checkpoints/alertable_V3/checkpoint_epoch_16.pt
🗂️  Label mapping      : /home/andres/Documentos/proyecto4geeks/data/interim/processed_dataset/label_mapping_alertableV2.pkl


## 3 · Cargar label mapping

In [48]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    """Devuelve (label2idx, idx2label, num_classes)."""
    with open(path, "rb") as f:
        payload = pickle.load(f)

    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}

    num_classes = len(label2idx)
    return label2idx, idx2label, num_classes


label2idx, idx2label, NUM_CLASSES = load_label_mapping(LABEL_MAPPING_PATH)

print(f"✅ {NUM_CLASSES} clases cargadas")
print("   Clases:", list(label2idx.keys()))

✅ 2 clases cargadas
   Clases: [False, True]


## 4 · Instanciar `Preprocess` (pipeline centralizado)

En lugar de construir los transforms manualmente, se delega en `Preprocess`.
Los parámetros deben ser **idénticos** a los usados en `preprocess.py → main()`.


In [49]:
# Instanciar Preprocess con los mismos parámetros que preprocess.py → main()
# target_duration=None → sin recorte (comportamiento de inferencia)
pp_config = PreprocessConfig(
    sample_rate=SAMPLE_RATE,
    target_duration=None,       # sin padding/recorte en inferencia
    normalize_peak=True,
    peak_target=PEAK_TARGET,
    n_mels=N_MELS,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    save_audio=False,
    save_mel=False,
    save_mfcc=False,
    save_waveform=False,
    augment_inference=AUGMENT_INFERENCE,
)

pp = Preprocess(config=pp_config)
device = pp.device

print(f"🖥️  Dispositivo: {device}")
print(f"🔊 AUGMENT_INFERENCE: {AUGMENT_INFERENCE}")
print("✅ Preprocess instanciado — transforms listos")

🖥️  Dispositivo: cuda
🔊 AUGMENT_INFERENCE: True
✅ Preprocess instanciado — transforms listos


## 5 · Pipeline de preprocesado — delegado en `Preprocess.process_audio_file()`

`Preprocess.process_audio_file(path)` aplica exactamente el mismo pipeline
que `preprocess.py → main()`: mono → resampleo → fix_length → normalización de pico → mel + mfcc.


In [50]:
def load_and_preprocess(
    audio_path: Path,
    augment_inference: bool = AUGMENT_INFERENCE,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Wrapper que delega en Preprocess.process_audio_file().
    Devuelve (mel, mfcc, waveform) con shape:
      mel      : [1, N_MELS, T]  float32
      mfcc     : [1, N_MFCC, T]  float32
      waveform : [1, T]           float32  — necesario para modo mel_waveform
    """
    mel, mfcc = pp.process_audio_file(audio_path, augment_inference=augment_inference)
    # Obtener waveform por separado para mel_waveform
    # process_audio_file ya aplica todo el pipeline; recalculamos solo el waveform
    import soundfile as sf
    import torchaudio.transforms as T

    audio_np, sr = sf.read(str(audio_path))
    waveform = torch.tensor(audio_np, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.transpose(0, 1)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.to(device)
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE).to(device)
        waveform = resampler(waveform)
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = (waveform / peak * PEAK_TARGET).float()

    return mel.float(), mfcc.float(), waveform


print("✅ load_and_preprocess() listo (delega en Preprocess)")

✅ load_and_preprocess() listo (delega en Preprocess)


## 6 · Cargar modelo

In [51]:
def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel      = any(k.startswith("cnn_mel.")      for k in keys)
    has_mfcc     = any(k.startswith("cnn_mfcc.")     for k in keys)
    has_waveform = any(k.startswith("cnn_wave.") for k in keys)

    if has_mel and has_waveform:
        return "mel_waveform"
    elif has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    else:
        raise ValueError("No se encontraron keys cnn_mel.*, cnn_mfcc.* ni waveform_cnn.* en el state_dict.")


def load_model(model_path: Path, num_classes: int, dropout: float) -> tuple[ImprovedMFCCCNN, str]:
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        epoch_info = checkpoint.get("epoch", "?")
        acc_info   = checkpoint.get("best_acc", "?")
        print(f"   📌 Checkpoint — epoch: {epoch_info}  |  best_acc: {acc_info}")
    else:
        state_dict = checkpoint

    detected_mode = detect_mode_from_state_dict(state_dict)
    print(f"   🔍 Modo detectado automáticamente: {detected_mode}")

    model = ImprovedMFCCCNN(num_classes=num_classes, dropout=dropout, mode=detected_mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, detected_mode


model, MODE = load_model(MODEL_PATH, NUM_CLASSES, DROPOUT)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo cargado  |  {total_params:,} parámetros  |  modo: {MODE}")

   📌 Checkpoint — epoch: 16  |  best_acc: 0.8626893939393939
   🔍 Modo detectado automáticamente: mel_waveform
✅ Modelo cargado  |  1,839,492 parámetros  |  modo: mel_waveform


## 7 · Función de predicción

In [52]:
@torch.inference_mode()
def predict(audio_path: Path, top_k: int = TOP_K) -> dict:
    """
    Retorna un dict con:
      - filename    : nombre del archivo
      - prediction  : clase predicha (bool — True=alertable, False=no alertable)
      - confidence  : probabilidad de la clase predicha (float)
      - top_k       : lista de (clase, prob) para las top_k predicciones
      - logits_raw  : tensor de logits (para debugging)
    """
    mel, mfcc, waveform = load_and_preprocess(audio_path)

    # Añadir dimensión de batch → [1, 1, F, T] / [1, 1, T]
    mel_b      = mel.unsqueeze(0)
    mfcc_b     = mfcc.unsqueeze(0)
    waveform_b = waveform.unsqueeze(0)

    if MODE == "mel_only":
        logits = model(mel=mel_b)
    elif MODE == "mel_mfcc":
        logits = model(mel=mel_b, mfcc=mfcc_b)
    elif MODE == "mel_waveform":
        logits = model(mel=mel_b, waveform=waveform_b)
    else:
        raise ValueError(f"Modo desconocido: {MODE}")

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()

    top_probs, top_idxs = probs.topk(min(top_k, len(idx2label)))

    pred_idx   = int(top_idxs[0])
    pred_label = idx2label[pred_idx]
    pred_conf  = float(top_probs[0])

    top_list = [(idx2label[int(i)], float(p)) for i, p in zip(top_idxs, top_probs)]

    return {
        "filename"   : audio_path.name,
        "prediction" : pred_label,
        "confidence" : pred_conf,
        "top_k"      : top_list,
        "logits_raw" : logits.squeeze(0).cpu(),
    }


print("✅ Función predict() lista")

✅ Función predict() lista


## 8 · Probar un audio individual (opcional)

In [53]:
# ── Cambia esto al archivo que quieras probar ───────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

# Demo: coge el primer audio de la carpeta si existe
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict(SINGLE_AUDIO)

    print(f"\n🔊 Archivo    : {result['filename']}")
    print(f"🏆 Predicción : {result['prediction']}")
    print(f"📊 Confianza  : {result['confidence']:.2%}")
    print(f"\n📋 Top-{TOP_K}:")
    for rank, (label, prob) in enumerate(result["top_k"], 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. {label:<25} {prob:.2%}  {bar}")
else:
    print(f"⚠️  No hay audios .wav/.mp3 en {AUDIO_FOLDER}")


🔊 Archivo    : 11325622-police-siren-sound-effect-240674.mp3
🏆 Predicción : True
📊 Confianza  : 97.79%

📋 Top-2:
  1. 1                         97.79%  █████████████████████████████
  2. 0                         2.21%  


## 9 · Inferencia por lotes sobre toda la carpeta

In [54]:
from tqdm import tqdm

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"📂 {len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows   = []
errors = []
prediction = {"ok": 0, "no_ok": 0}

for audio_path in tqdm(audios, desc="Procesando audios"):
    try:
        result = predict(audio_path)

        # Validación por estructura de carpetas:
        #   .../alertables/audio.wav   → esperamos True
        #   .../no_alertables/audio.wav → esperamos False
        folder_name = audio_path.parent.name.lower()
        if folder_name in ("alertables", "alertable"):
            expected = True
        elif folder_name in ("no_alertables", "no_alertable"):
            expected = False
        else:
            expected = None  # carpeta sin etiqueta conocida

        if expected is not None:
            if result["prediction"] == expected:
                prediction["ok"] += 1
            else:
                prediction["no_ok"] += 1

        row = {
            "audio_path"  : str(audio_path.parent.name),
            "filename"    : result["filename"],
            "prediction"  : result["prediction"],
            "confidence"  : result["confidence"],
            "expected"    : expected,
            "correct"     : (result["prediction"] == expected) if expected is not None else None,
        }
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)

    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"❌ Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

total_labeled = prediction["ok"] + prediction["no_ok"]
acc = prediction["ok"] / total_labeled if total_labeled > 0 else float("nan")

print(f"\n✅ Procesados : {len(rows)}  |  Errores: {len(errors)}")
print(f"🎯 Correctos  : {prediction['ok']} / {total_labeled}  ({acc:.2%})")
print(f"❌ Incorrectos: {prediction['no_ok']}")
results_df.head(40)

📂 35 audios encontrados en /home/andres/Documentos/proyecto4geeks/tests/audios



Procesando audios: 100%|██████████| 35/35 [00:00<00:00, 39.18it/s]


✅ Procesados : 35  |  Errores: 0
🎯 Correctos  : 29 / 35  (82.86%)
❌ Incorrectos: 6


,audio_path,filename,prediction,confidence,expected,correct,prob_True,prob_False
0,alertables,11325622-police-siren-sound-effect-240674.mp3,True,0.976888,True,True,0.9769,0.0231
1,alertables,ElevenLabs_A_6_to_7-year-old_child_crying_and_...,True,0.604534,True,True,0.6045,0.3955
2,alertables,audio_607a0.mp3,False,0.757529,True,False,0.2425,0.7575
3,alertables,child-crime-aw2xrhhk.wav,True,0.902315,True,True,0.9023,0.0977
4,alertables,dragon-studio-car-crash-sound-effect-376874.mp3,True,0.873574,True,True,0.8736,0.1264
5,alertables,dragon-studio-dog-barking-406629.mp3,True,0.996034,True,True,0.9960,0.0040
6,alertables,explosion-meme_dTCfAHs.mp3,True,0.942295,True,True,0.9423,0.0577
7,alertables,freesound_community-car-crash-edit-two-92001.mp3,True,0.507523,True,True,0.5075,0.4925
8,alertables,freesound_community-dog-barking-70772.mp3,True,0.919976,True,True,0.9200,0.0800
9,alertables,freesound_community-glass-shatter-7-95202.mp3,True,0.792778,True,True,0.7928,0.2072


## 10 · Resumen de predicciones

In [55]:
if not results_df.empty:
    summary = (
        results_df
        .groupby("prediction")
        .agg(
            count=("filename", "count"),
            avg_confidence=("confidence", "mean"),
            correct=("correct", lambda x: x.sum() if x.notna().any() else None),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary["avg_confidence"] = summary["avg_confidence"].map("{:.2%}".format)
    print("📊 Distribución de predicciones:")
    display(summary)

    # Archivos con confianza baja
    CONFIDENCE_THRESHOLD = 0.60
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n⚠️  {len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%}:")
        display(low_conf[["filename", "prediction", "confidence", "expected"]])

    # Falsos negativos (alertable predicho como no alertable)
    if "expected" in results_df.columns:
        fn = results_df[(results_df["expected"] == True) & (results_df["prediction"] == False)]
        fp = results_df[(results_df["expected"] == False) & (results_df["prediction"] == True)]
        if not fn.empty:
            print(f"\n🚨 Falsos negativos (alertable → no alertable): {len(fn)}")
            display(fn[["filename", "confidence"]].head(10))
        if not fp.empty:
            print(f"\n⚠️  Falsos positivos (no alertable → alertable): {len(fp)}")
            display(fp[["filename", "confidence"]].head(10))

📊 Distribución de predicciones:


,prediction,count,avg_confidence,correct
0,False,23,83.04%,17
1,True,12,86.49%,12



⚠️  4 audios con confianza < 60%:


,filename,prediction,confidence,expected
7,freesound_community-car-crash-edit-two-92001.mp3,True,0.507523,True
10,freesound_community-m4-assault-rifle-long-burs...,False,0.500315,True
11,loud-grating-scrape-of-a-iic5ipsu.wav,False,0.595260,True
20,Bowed-Bass-C2.wav,False,0.504287,False



🚨 Falsos negativos (alertable → no alertable): 6


,filename,confidence
2,audio_607a0.mp3,0.757529
10,freesound_community-m4-assault-rifle-long-burs...,0.500315
11,loud-grating-scrape-of-a-iic5ipsu.wav,0.595260
14,master_of_dreams_car_passing_by_city_03_186.mp3,0.649796
15,strategy-game-building-so-ih3jfxnx.wav,0.875061
17,zapsplat_nature_fire_flame_burst_flamethrower_...,0.860316


## 11 · Exportar resultados a CSV (opcional)

In [56]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Resultados guardados en: {OUTPUT_CSV}")
else:
    print("⚠️  No hay resultados para exportar.")

✅ Resultados guardados en: /home/andres/Documentos/proyecto4geeks/tests/audios/predictions.csv
